In [3]:
# [V31 최종 완성본] - 카메라 패닝 동적 추적 + 원근 압축(Foreshortening) 대응 길이 필터 완화
!fusermount -u /content/drive
!pip install ultralytics

from google.colab import drive
import cv2
import numpy as np
import os, time, unicodedata, csv
from pathlib import Path
from ultralytics import YOLO

drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = Path("/content/drive/MyDrive")
HOMEPLATE_MODEL_PATH = "/content/newhomeplate.pt"
BAT_MODEL_PATH = "/content/best_3.pt"
VIDEO_DIR = DRIVE_ROOT / "KBO_edit"
RESULT_VIDEOS_DIR = DRIVE_ROOT / "KBO_result_videos_v31_final"
os.makedirs(RESULT_VIDEOS_DIR, exist_ok=True)

hp_model = YOLO(HOMEPLATE_MODEL_PATH)
bat_model = YOLO(BAT_MODEL_PATH)

def extract_ground_truth(filename):
    normalized_name = unicodedata.normalize('NFC', Path(filename).stem)
    if "노스윙" in normalized_name: return "NO SWING"
    elif "스윙" in normalized_name: return "SWING"
    return "UNKNOWN"

def detect_handedness(hp_keypoints):
    if len(hp_keypoints) < 4: return None
    pt0, pt1, back_point = hp_keypoints[0], hp_keypoints[1], hp_keypoints[3]
    if pt0[0] <= 1 or pt1[0] <= 1 or back_point[0] <= 1: return None
    front_avg_x = (pt0[0] + pt1[0]) / 2
    return back_point[0] > front_avg_x

def analyze_checkswing(video_path, output_path):
    cap = cv2.VideoCapture(str(video_path))
    w, h = int(cap.get(3)), int(cap.get(4))
    fps = cap.get(5)
    out = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc('m', 'p', '4', 'v'), fps, (w, h))

    temp_frames = []
    hp_data_per_frame = []
    handedness_votes = []

    # 1. 영상 읽기 및 매 프레임 홈플레이트 추출
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        temp_frames.append(frame)

        hp_res = hp_model(frame, conf=0.3, verbose=False)
        if hp_res[0].keypoints is not None and len(hp_res[0].keypoints) > 0:
            k = hp_res[0].keypoints[0].xy.cpu().numpy()[0]
            if len(k) >= 4 and k[0][0] > 1:
                hp_data_per_frame.append(k)
                hv = detect_handedness(k)
                if hv is not None: handedness_votes.append(hv)
            else: hp_data_per_frame.append(None)
        else: hp_data_per_frame.append(None)

    if not any(k is not None for k in hp_data_per_frame):
        cap.release()
        out.release()
        return "UNDETERMINED"

    is_left = sum(handedness_votes) > len(handedness_votes) / 2 if handedness_votes else True
    hand_text = "Left" if is_left else "Right"

    # [핵심 보정 1] 패닝에 영향받지 않는 벡터 기반 절대 기울기 도출
    dx_list, dy_list = [], []
    for k in hp_data_per_frame:
        if k is not None:
            dx_list.append(k[1][0] - k[0][0])
            dy_list.append(k[1][1] - k[0][1])

    stable_dx = np.median(dx_list)
    stable_dy = np.median(dy_list)
    if abs(stable_dy) < 1e-6: stable_dy = 1e-6
    slope_ratio = stable_dx / stable_dy

    # [핵심 보정 2] 빈 프레임(가려짐) 좌표 채워넣기 (Forward & Backward Fill)
    filled_hp = []
    last_valid_k = None
    for k in hp_data_per_frame:
        if k is not None: last_valid_k = k
        filled_hp.append(last_valid_k)

    last_valid_k = None
    for i in range(len(filled_hp)-1, -1, -1):
        if filled_hp[i] is not None: last_valid_k = filled_hp[i]
        elif last_valid_k is not None: filled_hp[i] = last_valid_k

    # 2. 배트 추적 (동적 기준점 사용)
    SPATIAL_BUFFER = 250
    bat_trajectory = []

    for f_idx, frame in enumerate(temp_frames):
        current_hp = filled_hp[f_idx]
        if current_hp is None: continue

        # 현재 프레임의 진짜 홈플레이트 중심값 계산 (패닝 대응)
        current_mid_x = (current_hp[0][0] + current_hp[1][0]) / 2

        bat_res = bat_model(frame, conf=0.1, verbose=False)
        if bat_res[0].keypoints is not None and len(bat_res[0].keypoints) > 0:
            if bat_res[0].boxes and len(bat_res[0].boxes) > 0:
                idx = bat_res[0].boxes.conf.argmax().item()
                bk = bat_res[0].keypoints[idx].xy.cpu().numpy()[0]
                bb = bat_res[0].boxes.xyxy[idx].cpu().numpy()
                if len(bk) >= 2 and bk[0][0] > 1 and bk[1][0] > 1:
                    # 패닝된 현재 중심점 기준으로 버퍼 계산
                    if abs(((bb[0]+bb[2])/2) - current_mid_x) <= SPATIAL_BUFFER:
                        blen = np.sqrt(np.sum((bk[0]-bk[1])**2))
                        # [핵심 보정 3] 배트 길이 하한선 40으로 완화 (원근 압축 대응)
                        if 40 <= blen <= 350:
                            bat_trajectory.append({'f_idx': f_idx, 'p0': bk[0], 'p1': bk[1]})

    final_judge = "NO SWING"
    judge_f = -1
    consecutive = 0
    traj_dict = {b['f_idx']: b for b in bat_trajectory}

    for f_idx in range(len(temp_frames)):
        if f_idx in traj_dict:
            b = traj_dict[f_idx]
            p0, p1 = b['p0'], b['p1']

            if p1[1] > p0[1]:
                x_threshold = p0[0] + (p1[1] - p0[1]) * slope_ratio
                crossed = (p1[0] < x_threshold) if is_left else (p1[0] > x_threshold)

                if crossed:
                    consecutive += 1
                    if consecutive >= 2:
                        final_judge = "SWING"
                        if judge_f == -1: judge_f = f_idx
                else: consecutive = 0
            else: consecutive = 0
        else: consecutive = 0

    # 3. 시각화 (현재 프레임 홈플레이트 좌표 반영)
    for f_idx, frame in enumerate(temp_frames):
        overlay = frame.copy()
        current_hp = filled_hp[f_idx]

        if current_hp is not None:
            dyn_mid_x = (current_hp[0][0] + current_hp[1][0]) / 2
            dyn_mid_y = (current_hp[0][1] + current_hp[1][1]) / 2

            x_top_mid = int(dyn_mid_x + (0 - dyn_mid_y) * slope_ratio)
            x_bot_mid = int(dyn_mid_x + (h - dyn_mid_y) * slope_ratio)
            cv2.line(overlay, (x_top_mid, 0), (x_bot_mid, h), (0, 255, 255), 3)

        if f_idx in traj_dict:
            b = traj_dict[f_idx]
            bp0, bp1 = b['p0'].astype(int), b['p1'].astype(int)

            x_top_thresh = int(bp0[0] + (0 - bp0[1]) * slope_ratio)
            x_bot_thresh = int(bp0[0] + (h - bp0[1]) * slope_ratio)
            cv2.line(overlay, (x_top_thresh, 0), (x_bot_thresh, h), (255, 0, 255), 2)

            cv2.line(overlay, tuple(bp0), tuple(bp1), (0, 255, 255), 5)
            cv2.circle(overlay, tuple(bp0), 8, (0, 0, 255), -1)
            cv2.circle(overlay, tuple(bp1), 8, (0, 255, 0), -1)

        box_w, box_h = 600, 280
        cv2.rectangle(overlay, (w - box_w - 30, 30), (w - 30, box_h + 30), (0, 0, 0), -1)
        cv2.rectangle(overlay, (w - box_w - 30, 30), (w - 30, box_h + 30), (255, 255, 255), 2)

        font_style = cv2.FONT_HERSHEY_TRIPLEX
        cv2.putText(overlay, f"Batter: {hand_text}-handed", (w - box_w + 10, 80),
                    font_style, 1.4, (0, 255, 255), 2)

        cv2.circle(overlay, (w - box_w + 40, 130), 8, (0, 0, 255), -1)
        cv2.putText(overlay, ": p0 (Handle/Knob)", (w - box_w + 70, 145),
                    font_style, 1.2, (255, 255, 255), 1)

        cv2.circle(overlay, (w - box_w + 40, 185), 8, (0, 255, 0), -1)
        cv2.putText(overlay, ": p1 (Head/Tip)", (w - box_w + 70, 200),
                    font_style, 1.2, (255, 255, 255), 1)

        cv2.line(overlay, (w - box_w + 25, 245), (w - box_w + 55, 245), (255, 0, 255), 3)
        cv2.putText(overlay, ": Threshold Line", (w - box_w + 70, 255),
                    font_style, 1.2, (255, 255, 255), 1)

        if final_judge == "SWING":
            cv2.putText(overlay, "SWING!", (50, 110), font_style, 5.0, (0, 0, 255), 12)
            if f_idx >= judge_f and judge_f != -1:
                cv2.putText(overlay, ">> SYSTEM JUDGE POINT <<", (50, 220),
                            font_style, 2.2, (0, 255, 255), 5)
        else:
            cv2.putText(overlay, "NO SWING", (50, 110), font_style, 5.0, (0, 255, 0), 12)

        out.write(overlay)

    cap.release()
    out.release()
    return final_judge

# ========================================================
# 3. 메인 실행 루프
# ========================================================
video_files = sorted([f for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4')])
results_log = []
tp, tn, fp, fn = 0, 0, 0, 0

print("====================================")
print(" V31 논문용 최종 시스템 검증 ")
print(" (동적 홈플레이트 추적 및 길이 필터 완화 적용) ")
print("====================================")

for index, filename in enumerate(video_files):
    v_path = VIDEO_DIR / filename
    o_path = RESULT_VIDEOS_DIR / filename

    gt = extract_ground_truth(filename)
    pred = analyze_checkswing(v_path, o_path)

    match = (pred == gt)
    if gt == "SWING":
        if pred == "SWING": tp += 1
        else: fn += 1
    elif gt == "NO SWING":
        if pred == "NO SWING": tn += 1
        else: fp += 1

    print(f"[{index+1}/{len(video_files)}] {filename} | GT: {gt:8} | Pred: {pred:8} | {'O' if match else 'X'}")
    results_log.append({'filename': filename, 'ground_truth': gt, 'prediction': pred})

total = len(video_files)
acc = (tp + tn) / total if total > 0 else 0
prec = tp / (tp + fp) if (tp + fp) > 0 else 0
rec = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

print("\n==================================================")
print("FINAL RESULTS")
print("==================================================")
print(f"Accuracy:  {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1:.4f}")
print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}")
print("==================================================")

csv_path = RESULT_VIDEOS_DIR / "evaluation_report_v31_final.csv"
with open(csv_path, 'w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['filename', 'ground_truth', 'prediction'])
    writer.writeheader()
    writer.writerows(results_log)

Mounted at /content/drive
 V31 논문용 최종 시스템 검증 
 (동적 홈플레이트 추적 및 길이 필터 완화 적용) 
[1/145] 250819_SSGKT_8초_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | O
[2/145] 250820_SSGKT_8말_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | O
[3/145] 250820_롯데LG_9초_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | O
[4/145] 250820_삼성NC_9초_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | O
[5/145] 250820_키움KIA_7초_스윙.mp4 | GT: SWING    | Pred: NO SWING | X
[6/145] 250821_SSGKT_5말_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | O
[7/145] 250821_SSGKT_9말_스윙.mp4 | GT: SWING    | Pred: SWING    | O
[8/145] 250821_롯데LG_7말_스윙.mp4 | GT: SWING    | Pred: SWING    | O
[9/145] 250821_삼성NC_8초_스윙.mp4 | GT: SWING    | Pred: SWING    | O
[10/145] 250821_키움KIA_4말_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | O
[11/145] 250822_KT두산_8초_스윙.mp4 | GT: SWING    | Pred: NO SWING | X
[12/145] 250822_LGKIA_1말_노스윙.mp4 | GT: NO SWING | Pred: NO SWING | O
[13/145] 250822_LGKIA_4초_스윙